In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/crsthk.xyz",
                 sep='\s+', header=None,
                 names=['lon', 'lat', 'thickness'])

In [3]:
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nLon range:  {df['lon'].min():.2f} – {df['lon'].max():.2f}")
print(f"Lat range:  {df['lat'].min():.2f} – {df['lat'].max():.2f}")
print(f"Value range: {df['thickness'].min():.2f} – {df['thickness'].max():.2f}")
print(f"Mean: {df['thickness'].mean():.2f}")
print(f"Zero/negative values: {(df['thickness'] <= 0).sum()}")
print(f"\nResolution:")
print(np.sort(np.unique(np.diff(np.sort(df['lon'].unique()))))[:3])


Shape: (64800, 3)

First few rows:
     lon   lat  thickness
0 -179.5  89.5       8.06
1 -178.5  89.5       8.08
2 -177.5  89.5       8.08
3 -176.5  89.5       8.09
4 -175.5  89.5       8.09
5 -174.5  89.5       8.10
6 -173.5  89.5       8.11
7 -172.5  89.5       8.08
8 -171.5  89.5       8.05
9 -170.5  89.5       8.04

Lon range:  -179.50 – 179.50
Lat range:  -89.50 – 89.50
Value range: 4.13 – 80.00
Mean: 21.01
Zero/negative values: 0

Resolution:
[1.]


In [4]:
# Patch level
patches = {
    "Kanto_Japan":      (138.5, 34.5, 141.5, 37.2),
    "Tohoku_Japan":     (140.5, 37.5, 143.5, 40.5),
    "Central_Chile":    (-72.5, -36.5, -69.5, -33.5),
    "Central_Turkey":   (35.5, 36.5, 39.0, 39.0),
    "Nepal":            (83.5, 27.0, 86.5, 29.7),
    "North_Island_NZ":  (174.5, -40.5, 178.0, -37.5),
    "Sumatra":          (100.5, -5.5, 104.5, -2.0),
    "Kutch_India":      (68.5, 21.5, 72.0, 24.5),
    "Sichuan_China":    (102.0, 29.5, 105.5, 32.5),
    "W_Australia":      (117.0, -32.0, 120.5, -29.0),
    "S_Norway":         (5.0, 58.5, 9.0, 61.5),
    "Ordos_China":      (107.5, 37.0, 111.0, 40.0),
}

print(f"\n{'Patch':<25} {'N cells':>7} {'Min':>8} {'Max':>8} {'Mean':>8}")
print("-" * 60)
for name, (minlon, minlat, maxlon, maxlat) in patches.items():
    patch = df[
        (df['lon'] >= minlon) & (df['lon'] <= maxlon) &
        (df['lat'] >= minlat) & (df['lat'] <= maxlat)
    ]
    if len(patch) > 0:
        t = patch['thickness']
        print(f"{name:<25} {len(patch):>7} {t.min():>8.2f} {t.max():>8.2f} {t.mean():>8.2f}")
    else:
        print(f"{name:<25}  NO DATA IN BOUNDS")


Patch                     N cells      Min      Max     Mean
------------------------------------------------------------
Kanto_Japan                    12    10.89    31.43    24.42
Tohoku_Japan                   16    11.36    30.00    22.99
Central_Chile                  16     7.53    54.33    39.59
Central_Turkey                 12    32.78    39.75    36.51
Nepal                          12    42.00    72.04    55.64
North_Island_NZ                16    19.28    40.67    28.99
Sumatra                        20     7.03    33.00    25.56
Kutch_India                    16    22.05    38.28    32.74
Sichuan_China                  16    38.98    61.20    47.63
W_Australia                    12    34.36    40.18    36.66
S_Norway                       16    29.84    39.59    34.15
Ordos_China                    12    38.99    44.00    42.46


## Insights

<p>The crustal thickness layer is sourced from CRUST1.0, sharing the same 1° resolution global grid as the sediment thickness layer  - 64,800 cells, identical spatial extent and coordinate system. Values represent Moho depth in kilometres, ranging from approximately 7 km (oceanic crust at subduction trenches) to 72 km (thickened continental crust beneath the Himalayas). No nodata or sentinel values were identified. The same interpolation caveat from the sediment layer applies here: 12–20 cells per patch means significant smoothing when resampling to 0.1°.</p>

<p>Patch-level means are physically precise and ordered exactly as crustal physics predicts, which provides strong independent validation that the layer is correct and will contribute meaningful discriminative signal to the frozen geological prior. Nepal records the deepest mean Moho at 55.64 km, with a maximum of 72.04 km consistent with published estimates for the central Himalayan collision zone where the Indian plate is being actively underthrust beneath the Tibetan Plateau. Sichuan follows at 47.63 km, reflecting the eastern margin of the same collision-thickened crustal system. At the other end, Kanto and Tohoku return the shallowest means (24.42 and 22.99 km respectively), with minimums around 11 km that capture genuine oceanic crust at the trench end of these patches. The large within-patch range in both Japanese patches reflects the transition from thin oceanic to thicker continental crust across the subduction interface.</p>

<p>The stable craton and shield patches: Western Australia (36.66 km), Ordos (42.46 km), and Norway (34.15 km), all show moderate crustal thickness with notably narrow min-max ranges. This internal consistency is itself a geologically meaningful signal: tectonic stability is expressed not just in the mean Moho depth but in the absence of lateral crustal heterogeneity within the patch. This contrast with the high-variance subduction patches will be a useful discriminator in the clustering step.</p>

<p>Chile shows the largest within-patch variance of any patch, with a minimum of 7.53 km (oceanic crust at the Nazca-South American trench) and a maximum of 54.33 km (thickened Andean crust on the eastern side). This reflects the full arc-trench system being captured within a single 300×300 km patch. It is geologically rich but worth flagging as a patch where cell-level predictions may vary dramatically depending on position relative to the trench. Sumatra shows a similar but less extreme pattern, with its 7.03 km minimum confirming oceanic crust sampling consistent with the patch's ~60% ocean fraction.</p>

<p>Taken together with the sediment thickness and Vs30 layers already inspected, a coherent multi-layer picture of geological regime is emerging across the 12 patches. The three collision/thrust patches (Nepal, Sichuan, Ordos) consistently show deep Moho and thick sediment cover. The subduction patches (Kanto, Tohoku, Sumatra) show thin crust with high internal variance. The stable cratons (Australia, Norway) show moderate Moho depth, near-zero sediment, and low internal variance. This three-way separation in the static feature space is an early positive signal that the GMM clustering will recover interpretable and physically meaningful regime boundaries.</p>